# Baseline Model Training With Large Sample Data (API)

This notebook trains only the baseline `taiko_transformer` using the direct Python API.

Flow:
1. Build persistent dataset artifacts from `sample_data_large/raw`.
2. Create or resume the baseline training context.
3. Train the baseline model and inspect the resulting checkpoint paths.


In [ ]:
from pathlib import Path

import torch

from src.model import (
    ArchitectureSpec,
    TrainingSpec,
    WandbConfig,
    build_training_artifacts,
    create_training_context,
    load_training_context_from_checkpoint,
    prepare_sample_data_artifacts,
    train_context,
)

repo_root = Path.cwd()
raw_osz_dir = repo_root / "sample_data_large" / "raw"
data_root = repo_root / "sample_data_large"
training_dir = data_root / "training"
checkpoints_dir = repo_root / "checkpoints" / "sample_large_baseline"
last_checkpoint = checkpoints_dir / "last.ckpt"
best_checkpoint = checkpoints_dir / "best.ckpt"

index_cache_dir = training_dir / "index_cache"
inference_snapshots_dir = checkpoints_dir / "inference_snapshots"

epochs = 10
batch_size = 8
num_workers = 0
learning_rate = 1e-3
architecture_name = "taiko_transformer"
prepare_data = False
save_inference_every_n_steps = 1000
use_resume_if_available = True
use_wandb = False
wandb_log_every_batches = 100
wandb_notebook_name = "train_baseline_largesample_data_api.ipynb"
wandb_api_key = ""
wandb_offline = False

if torch.cuda.is_available():
    best_device = "cuda"
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    best_device = "mps"
else:
    best_device = "cpu"

checkpoints_dir.mkdir(parents=True, exist_ok=True)

print(f"repo_root         : {repo_root}")
print(f"raw_osz_dir       : {raw_osz_dir}")
print(f"data_root         : {data_root}")
print(f"training_dir      : {training_dir}")
print(f"checkpoints_dir   : {checkpoints_dir}")
print(f"last_checkpoint   : {last_checkpoint}")
print(f"index_cache_dir   : {index_cache_dir}")
print(f"inference_snapshots_dir   : {inference_snapshots_dir}")
print(f"best_checkpoint   : {best_checkpoint}")
print(f"best_device       : {best_device}")


## Step 1: Prepare persistent dataset artifacts


In [ ]:
if prepare_data:
    baseline_artifacts = prepare_sample_data_artifacts(
        osz_inputs=[str(raw_osz_dir)],
        data_root=data_root,
    )
else:
    baseline_artifacts = build_training_artifacts(data_root, checkpoints_dir=checkpoints_dir)
    print("Skipping raw-data preparation and reusing existing training artifacts.")

print(baseline_artifacts)


## Step 2: Create or resume the baseline training context


In [ ]:
baseline_architecture_spec = ArchitectureSpec(name=architecture_name)

baseline_training_spec = TrainingSpec(
    epochs=epochs,
    batch_size=batch_size,
    num_workers=num_workers,
    lr=learning_rate,
    device=best_device,
)

wandb_config = None
if use_wandb:
    wandb_config = WandbConfig(
        enabled=True,
        run_name="baseline_large_sample",
        log_every_n_batches=wandb_log_every_batches,
        notebook_name=wandb_notebook_name,
        offline=wandb_offline,
        api_key=wandb_api_key,
        mode_name_for_run=architecture_name,
    )

if use_resume_if_available and last_checkpoint.exists():
    baseline_context = load_training_context_from_checkpoint(
        last_checkpoint,
        data_root=data_root,
        device=best_device,
        batch_size=batch_size,
        num_workers=num_workers,
        checkpoints_dir=checkpoints_dir,
        index_cache_dir=index_cache_dir,
    )
    print(f"Resuming from checkpoint: {last_checkpoint}")
else:
    baseline_context = create_training_context(
        data_root=data_root,
        architecture_spec=baseline_architecture_spec,
        training_spec=baseline_training_spec,
        checkpoints_dir=checkpoints_dir,
        index_cache_dir=index_cache_dir,
    )
    print("Starting a fresh baseline training run.")

print("baseline architecture:", baseline_context.architecture_spec)
print("baseline ignore_index:", baseline_context.dataset.label_ignore_index)
print("start_epoch:", baseline_context.start_epoch)
print("target_epochs:", epochs)


## Step 3: Train the baseline model


In [ ]:
baseline_context = train_context(
    baseline_context,
    epochs=epochs,
    log_every_n_batches=wandb_log_every_batches,
    wandb_config=wandb_config,
    save_inference_every_n_steps=save_inference_every_n_steps,
    inference_snapshots_dir=inference_snapshots_dir,
)

assert last_checkpoint.exists(), "Checkpoint was not written after the baseline API run"

print("Training finished.")
print(f"last checkpoint: {last_checkpoint.resolve()}")
print(f"best checkpoint: {best_checkpoint.resolve()}")
print(f"inference snapshots dir: {inference_snapshots_dir.resolve()}")


## Optional inspection


In [ ]:
print("history keys:", baseline_context.history.keys())
print("training dir:", training_dir)
print("vocab json:", training_dir / "vocab.json")
print("splits json:", training_dir / "splits.json")
